# Notebook 05 — Margin Mechanics and SIMM

**quant-desk-toolkit** · github.com/hyun-quant/quant-desk-toolkit

---

Margin is the collateral exchanged between counterparties to reduce counterparty credit risk. This notebook covers both **Variation Margin (VM)** and **Initial Margin (IM)**, including the ISDA SIMM methodology.

1. **Schedule IM**: simplified notional-based IM (for sub-threshold counterparties)
2. **SIMM FX Delta**: ISDA SIMM IM for FX positions
3. **VM Engine**: simulating day-to-day VM calls under a CSA
4. **IM decay profiles**: how outstanding IM evolves as the portfolio ages
5. **MVA calculation**: cost of funding IM over time

**Prerequisites**: `instruments.pkl`, `exposure_results.pkl`, `xva_results.pkl`
**Outputs**: `margin_results.pkl`


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from margin import CSATerms, VMEngine, MarginEngine, simm_fx_delta

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

with open('instruments.pkl', 'rb') as f:
    inst = pickle.load(f)
with open('exposure_results.pkl', 'rb') as f:
    exp = pickle.load(f)
with open('xva_results.pkl', 'rb') as f:
    xva_data = pickle.load(f)

time_grid  = exp['time_grid']
EE         = exp['EE']
ENE        = exp['ENE']
mtm_matrix = exp['mtm_matrix']
mean_mtm   = mtm_matrix.mean(axis=0)   # representative path for VM simulation

print('Setup OK')
print(f'time_grid: {time_grid.shape},  MTM matrix: {mtm_matrix.shape}')

---
## 1. Schedule Initial Margin

The **Schedule IM** (BCBS-IOSCO simplified approach) uses a fixed percentage of notional, applying netting benefits via the NGR (Net-to-Gross Ratio):

$$IM_{schedule} = \text{Notional} \times \text{Percentage}(\text{asset class, maturity})$$

Typical percentages: 1% for <1Y IR, up to 15% for equity >5Y. Netting benefit: $IM_{net} = 0.4 \times IM_{gross} + 0.6 \times NGR \times IM_{gross}$


In [ ]:
# Schedule IM percentages (BCBS-IOSCO Annex A)
schedule_pct = {
    ('IR',     'short'):  0.01,   # < 1Y
    ('IR',     'medium'): 0.02,   # 1–5Y
    ('IR',     'long'):   0.04,   # > 5Y
    ('Credit', 'short'):  0.02,
    ('Credit', 'medium'): 0.05,
    ('Credit', 'long'):   0.10,
    ('Equity', 'short'):  0.06,
    ('Equity', 'medium'): 0.08,
    ('Equity', 'long'):   0.10,
    ('FX',     'short'):  0.04,
    ('FX',     'medium'): 0.06,
    ('FX',     'long'):   0.08,
}

notional = inst['swap_notional']
tenor    = inst['swap_tenor']
bucket   = 'medium' if 1 <= tenor <= 5 else ('short' if tenor < 1 else 'long')
pct      = schedule_pct[('IR', bucket)]

im_gross = notional * pct
print(f'Swap: $10mm, {tenor}Y IR ({bucket} bucket), Schedule pct = {pct*100:.1f}%')
print(f'Schedule IM (gross)  : ${im_gross:>12,.0f}')

# Netting benefit with a hypothetical receiver (offsetting) swap
im_gross_recv = notional * pct   # same size receiver swap
ngr_with_offset = 0.0            # perfect offset → NGR ≈ 0
im_net_offset = (0.4 * (im_gross + im_gross_recv) +
                 0.6 * ngr_with_offset * (im_gross + im_gross_recv))
print(f'\nWith opposite receiver swap (perfect offset):')
print(f'  IM gross (sum)       : ${im_gross + im_gross_recv:>12,.0f}')
print(f'  NGR                  : {ngr_with_offset:.2f}')
print(f'  IM net               : ${im_net_offset:>12,.0f}  (significant netting benefit)')

In [ ]:
# Schedule IM by notional and tenor
fig, ax = plt.subplots(figsize=(10, 4))
tenors_plot = np.array([0.5, 1, 2, 3, 5, 7, 10])
pcts_ir = []
for T in tenors_plot:
    b = 'medium' if 1 <= T <= 5 else ('short' if T < 1 else 'long')
    pcts_ir.append(schedule_pct[('IR', b)] * 100)

ax.bar(range(len(tenors_plot)), pcts_ir, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_xticks(range(len(tenors_plot)))
ax.set_xticklabels([f'{t}Y' for t in tenors_plot])
ax.set_title('Schedule IM Percentage: Interest Rate Asset Class')
ax.set_xlabel('Tenor bucket'); ax.set_ylabel('IM as % of Notional')
plt.tight_layout(); plt.savefig('nb05_schedule_im.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 2. SIMM FX Delta IM

ISDA SIMM computes IM from risk sensitivities (greeks), not notionals. For FX positions, the delta sensitivity is the notional in foreign currency, and IM = RW × |sensitivity|.

SIMM FX Risk Weight = 7.4% (SIMM v2.5)


In [ ]:
# FX delta sensitivities: net DV01-equivalent in each currency pair
# A $10mm EURUSD position has ~$10mm FX delta sensitivity
fx_sensitivities = {
    'EURUSD': 10_000_000,    # long $10mm EUR vs USD
    'GBPUSD':  5_000_000,    # long $5mm GBP vs USD
    'USDJPY': -3_000_000,    # short $3mm USD vs JPY
}

im_fx = simm_fx_delta(fx_sensitivities)
print('SIMM FX Delta IM:')
for pair, sens in fx_sensitivities.items():
    rw_contrib = 0.074 * abs(sens)
    print(f'  {pair}: sensitivity ${sens:>12,.0f}  →  RW contribution ${rw_contrib:>12,.0f}')
print(f'  Total FX SIMM IM (quadratic sum): ${im_fx:>12,.0f}')
print(f'  (= sqrt(sum of (RW × sens)²))')

---
## 3. VM Engine — Variation Margin Simulation

The `VMEngine` simulates day-to-day VM calls along the mean MTM path. When the MTM moves beyond the threshold + MTA, a margin call is triggered. The resulting `vm_net` tracks the running collateral balance.


In [ ]:
# Scenario A: zero-threshold CSA (full collateralisation)
csa_zero = CSATerms(threshold_them=0.0, threshold_we=0.0, mta_them=50_000, mta_we=50_000)
vm_engine_zero = VMEngine(csa_zero)
vm_res_zero = vm_engine_zero.simulate(mean_mtm, time_grid)
vm_net_zero  = vm_res_zero['vm_net']

# Scenario B: $500K threshold CSA (standard corporate)
csa_500k = CSATerms(threshold_them=500_000, threshold_we=500_000, mta_them=100_000, mta_we=100_000)
vm_engine_500k = VMEngine(csa_500k)
vm_res_500k = vm_engine_500k.simulate(mean_mtm, time_grid)
vm_net_500k  = vm_res_500k['vm_net']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(time_grid, mean_mtm/1e6, lw=2, color='steelblue', label='Swap MTM (mean path)')
ax.plot(time_grid, vm_net_zero/1e6, lw=2, ls='--', color='tomato', label='VM (zero threshold)')
ax.plot(time_grid, vm_net_500k/1e6, lw=2, ls=':', color='green', label='VM ($500K threshold)')
ax.axhline(y=0, color='black', lw=0.8)
ax.set_title('VM Collateral vs Swap MTM (Mean Path)')
ax.set_xlabel('Time (years)'); ax.set_ylabel('$mm'); ax.legend()

ax2 = axes[1]
exposure_zero  = np.maximum(mean_mtm - vm_net_zero, 0)
exposure_500k  = np.maximum(mean_mtm - vm_net_500k, 0)
exposure_uncoll = np.maximum(mean_mtm, 0)
ax2.plot(time_grid, exposure_uncoll/1e6, lw=2, color='tomato', label='Uncollateralised')
ax2.plot(time_grid, exposure_zero/1e6, lw=2, ls='--', color='steelblue', label='Zero threshold')
ax2.plot(time_grid, exposure_500k/1e6, lw=2, ls=':', color='green', label='$500K threshold')
ax2.set_title('Net Exposure After VM Collateral'); ax2.legend()
ax2.set_xlabel('Time (years)'); ax2.set_ylabel('Net exposure ($mm)')

plt.tight_layout(); plt.savefig('nb05_vm_sim.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 4. IM Profile Decay

As the portfolio ages, the outstanding IM decreases because remaining tenor (and therefore residual risk) shrinks. The `MarginEngine.im_profile()` provides two decay shapes:
- **Linear**: IM decays proportionally to remaining life $IM(t) = IM_0 \cdot (T-t)/T$
- **Sqrt**: front-loaded decay, appropriate when DV01 concentrates near maturity


In [ ]:
schedule_im = im_gross   # from earlier calculation

margin_eng = MarginEngine(csa=CSATerms(), schedule_im_amount=schedule_im)

im_linear = margin_eng.im_profile(time_grid, decay='linear', portfolio_maturity=5.0)
im_sqrt   = margin_eng.im_profile(time_grid, decay='sqrt',   portfolio_maturity=5.0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_grid, im_linear/1e6, lw=2, color='steelblue', label='Linear decay')
ax.plot(time_grid, im_sqrt/1e6, lw=2, ls='--', color='tomato', label='Sqrt decay (front-loaded)')
ax.set_title('IM Profile Over Time: Linear vs Sqrt Decay')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Outstanding IM ($mm)'); ax.legend()
plt.tight_layout(); plt.savefig('nb05_im_decay.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 5. MVA — Margin Valuation Adjustment

In [ ]:
from xva import MVAEngine

funding_spread = xva_data['funding_spread']  # 50bp

mva_linear = MVAEngine(im_profile=im_linear, time_grid=time_grid, funding_spread=funding_spread).compute()
mva_sqrt   = MVAEngine(im_profile=im_sqrt,   time_grid=time_grid, funding_spread=funding_spread).compute()

print(f'MVA (funding spread = {funding_spread*10000:.0f}bp):')
print(f'  Linear decay IM  →  MVA = ${mva_linear:>10,.0f}')
print(f'  Sqrt decay IM    →  MVA = ${mva_sqrt:>10,.0f}')
print()
print(f'Intuition: ${schedule_im:,.0f} IM × {funding_spread*100:.2f}% spread × ~{5/2:.1f}Y avg life')
print(f'  = ${schedule_im * funding_spread * 5/2:,.0f}  (rough approximation)')

---
## 6. Full MarginEngine: VM + IM Combined

In [ ]:
csa_full = CSATerms(threshold_them=500_000, mta_them=100_000)
margin_full = MarginEngine(csa=csa_full, schedule_im_amount=schedule_im)
result_full = margin_full.compute(mean_mtm, time_grid, portfolio_maturity=5.0, im_decay='linear')

print('MarginEngine.compute() output keys:', list(result_full.keys()))
print(f'  IM_initial      : ${result_full["IM_initial"]:>12,.0f}')
print(f'  IM_profile[0]   : ${result_full["IM_profile"][0]:>12,.0f}  (at t=0)')
print(f'  IM_profile[-1]  : ${result_full["IM_profile"][-1]:>12,.0f}  (at maturity)')
print(f'  VM at t=0       : ${result_full["vm_result"]["vm_net"][0]:>12,.0f}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_grid, result_full['IM_profile']/1e6, lw=2, color='steelblue', label='IM profile (linear)')
ax.plot(time_grid, result_full['vm_result']['vm_net']/1e6, lw=2, ls='--', color='tomato', label='VM (mean path)')
ax.plot(time_grid, result_full['total_collateral_us']/1e6, lw=2, ls=':', color='green', label='Total collateral (IM + VM)')
ax.set_title('Combined IM + VM Collateral Profile')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Collateral ($mm)'); ax.legend()
plt.tight_layout(); plt.savefig('nb05_combined_margin.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 7. Save Outputs

In [ ]:
margin_results = {
    'schedule_im'    : schedule_im,
    'im_gross'       : im_gross,
    'im_fx'          : im_fx,
    'vm_net_zero'    : vm_net_zero,
    'vm_net_500k'    : vm_net_500k,
    'im_linear'      : im_linear,
    'im_sqrt'        : im_sqrt,
    'mva_linear'     : mva_linear,
    'mva_sqrt'       : mva_sqrt,
    'margin_full'    : result_full,
    'funding_spread' : funding_spread,
    'time_grid'      : time_grid,
}
with open('margin_results.pkl', 'wb') as f:
    pickle.dump(margin_results, f)
print('Saved margin_results.pkl')
print('Proceed to: notebook_06_greeks.ipynb')